In [67]:
log_data = []

import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_json(r'..\data\raw\streaming_users_dirty.json')

#Tamaño original
df.shape
log_data.append({
    "Paso": "01", 
    "Descripción": "Dataset original cargado", 
    "Filas": df.shape[0], 
    "Nulos": int(df.isnull().sum().sum()), 
    "Retención (%)": 100.0})

Eliminación de duplicados por user_id

El bloque de abajo calcula cuántos duplicados hay en el dataset comparando la cantidad total de filas contra la cantidad de valores únicos en user_id, y lo imprime para tener una referencia previa. Después usa drop_duplicates con subset=['user_id'] y keep='first' para quedarse con la primera aparición de cada usuario y descartar el resto, y vuelve a calcular la diferencia para confirmar cuántos registros se sacaron.

In [68]:
# Verificar duplicados antes
duplicados = len(df) - df['user_id'].nunique()
print(f"Duplicados encontrados: {duplicados}")

# Eliminar duplicados por user_id (mantener el primero)
df = df.drop_duplicates(subset=['user_id'], keep='first')

# Verificar después
duplicados_eliminados = len(df) - df['user_id'].nunique()
print(f"Duplicados eliminados: {duplicados - duplicados_eliminados}")

log_data.append({ "Paso": "02", "Descripción": "Eliminación de duplicados por user_id (keep=first)", "Filas": df.shape[0], "Nulos": int(df.isnull().sum().sum()), "Retención (%)": round((df.shape[0] / 18000) * 100, 2)
})

df.to_json('../data/processed/streaming_users_clean.json', 
           orient='records', 
           indent=2,
           date_format='iso',
           force_ascii=False)

log_df = pd.DataFrame(log_data)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

Duplicados encontrados: 160
Duplicados eliminados: 160


Análisis de valores nulos 

Luego de eliminados los duplicados nos centramos en los nulos. EL siguiente codigo  hace un diagnóstico completo de los datos faltantes en el dataset antes de decidir cómo tratarlos

In [69]:
df = pd.read_json('../data/processed/streaming_users_clean.json')

print("=== VALORES NULOS POR COLUMNA ===\n")
nulos_por_columna = df.isnull().sum()
print(nulos_por_columna)

# Porcentaje de nulos por columna
print("\n=== PORCENTAJE DE NULOS POR COLUMNA ===\n")
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100
print(porcentaje_nulos[porcentaje_nulos > 0].round(2))

# Total de nulos en todo el dataset
print(f"\n=== TOTAL DE NULOS EN EL DATASET ===\n")
print(f"Total de valores nulos: {df.isnull().sum().sum()}")

# Mostrar filas donde al menos una columna tiene nulo
registros_con_nulos = df[df.isnull().any(axis=1)]
print(f"Total de registros con al menos un nulo: {len(registros_con_nulos)}")

=== VALORES NULOS POR COLUMNA ===

user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins     193
country                       0
favorite_genre              240
last_login_date             320
customer_support_tickets      0
dtype: int64

=== PORCENTAJE DE NULOS POR COLUMNA ===

monthly_watch_time_mins    2.41
favorite_genre             3.00
last_login_date            4.00
dtype: float64

=== TOTAL DE NULOS EN EL DATASET ===

Total de valores nulos: 753
Total de registros con al menos un nulo: 737


Estandarización y limpieza de fechas en la columna "last_login_date"

Este bloque toma la columna last_login_date, que viene con fechas en varios formatos distintos (a veces día-mes-año, a veces mes-día-año, a veces con guiones, a veces con barras), y las pasa todas a un formato único DD/MM/YYYY probando una lista de formatos posibles hasta encontrar el que matchea; si ninguno funciona o el valor está vacío, lo marca como "--/--/--". Después convierte esas fechas ya estandarizadas a tipo date para poder operar con ellas, y usa eso para detectar y eliminar registros con fechas de login posteriores al 06/07/2026 (que no tendrían sentido, ya que serían fechas futuras respecto a la fecha límite del análisis).

In [70]:
df = pd.read_json('../data/processed/streaming_users_clean.json')

# PASO 1: Función para estandarizar fechas
def estandarizar_fecha(fecha):
    """Convierte fechas a formato DD/MM/YYYY o devuelve --/--/--"""
    if pd.isna(fecha) or fecha == 'not_available' or fecha == '':
        return '--/--/--'
    
    # Lista de formatos posibles
    formatos = [
        '%Y-%m-%d',      # 2025-03-04
        '%m-%d-%Y',      # 11-19-2018
        '%d-%m-%Y',      # 11-05-2019
        '%Y/%m/%d',      # 2022/07/08
        '%d/%m/%Y',      # 11/12/2023
        '%m/%d/%Y',      # 01/06/2020
        '%d-%m-%Y'       # 04-03-2021
    ]
    
    for formato in formatos:
        try:
            fecha_dt = pd.to_datetime(fecha, format=formato)
            return fecha_dt.strftime('%d/%m/%Y')
        except:
            continue
    
    # Si ningún formato funciona, devolver --/--/--
    return '--/--/--'

# PASO 2: Aplicar estandarización
df['last_login_date'] = df['last_login_date'].apply(estandarizar_fecha)

# PASO 3: Convertir a datetime
def convertir_a_date(fecha_str):
    """Convierte string a date (solo fecha, sin hora)"""
    if fecha_str == '--/--/--':
        return pd.NaT
    try:
        return pd.to_datetime(fecha_str, format='%d/%m/%Y').date()
    except:
        return pd.NaT

df['last_login_date'] = df['last_login_date'].apply(convertir_a_date)

# PASO 4: Eliminar fechas futuras (después del 06/07/2026)
fecha_limite = datetime(2026, 7, 6).date()

print("=== ELIMINANDO FECHAS FUTURAS ===\n")
print(f"Fecha límite: {fecha_limite.strftime('%d/%m/%Y')}")

# Identificar fechas futuras
fechas_futuras = df[df['last_login_date'] > fecha_limite]
print(f"Registros con fechas futuras: {len(fechas_futuras)}")

if len(fechas_futuras) > 0:
    print("\nEjemplos de fechas futuras:")
    
    # Eliminar registros con fechas futuras
    df = df[df['last_login_date'] <= fecha_limite]
    print(f"\n✅ Registros eliminados: {len(fechas_futuras)}")

# PASO 5: Reemplazar NaT por --/--/---
# Convertir a string y reemplazar NaT
df['last_login_date'] = df['last_login_date'].apply(
    lambda x: '--/--/--' if pd.isna(x) else x.strftime('%d/%m/%Y')
)

# PASO 6: Verificar resultados
print("\n=== VERIFICACIÓN FINAL ===\n")
print(f"Total de registros: {len(df)}")

# Contar cuántos tienen --/--/--
nulos_final = (df['last_login_date'] == '--/--/--').sum()
print(f"Registros con --/--/--: {nulos_final}")

# Verificar que no haya fechas futuras
fechas_validas = df[df['last_login_date'] != '--/--/--']
print(f"\nFechas válidas (no --/--/--): {len(fechas_validas)}")

log_data.append({ "Paso": "03", "Descripción": "Estandarización y limpieza de fechas", "Filas": df.shape[0], "Nulos": int(df.isnull().sum().sum()), "Retención (%)": round((df.shape[0] / 18000) * 100, 2)
})

df.to_json('../data/processed/streaming_users_clean.json', 
           orient='records', 
           indent=2,
           date_format='iso',
           force_ascii=False)

log_df = pd.DataFrame(log_data)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

=== ELIMINANDO FECHAS FUTURAS ===

Fecha límite: 06/07/2026
Registros con fechas futuras: 15

Ejemplos de fechas futuras:

✅ Registros eliminados: 15

=== VERIFICACIÓN FINAL ===

Total de registros: 7601
Registros con --/--/--: 0

Fechas válidas (no --/--/--): 7601


filtrar los minutos de visualización al mes

Este bloque primero hace un diagnóstico rápido del estado de la columna monthly_watch_time_mins, mostrando cuántos registros hay en total, si hay valores nulos, y el mínimo, máximo y promedio actuales — esto sirve para tener una foto de "antes" y poder comparar después. Luego aplica un filtro que se queda solo con las filas donde el tiempo de visualización esté entre 0 y 5000 minutos, descartando así valores negativos (que no tienen sentido lógico) o valores extremadamente altos que probablemente sean outliers

In [71]:
df = pd.read_json('../data/processed/streaming_users_clean.json')

# Mostrar estado actual
print("=== ESTADO ACTUAL ===\n")
print(f"Total: {len(df)}")
print(f"Nulos: {df['monthly_watch_time_mins'].isnull().sum()}")
print(f"Mínimo: {df['monthly_watch_time_mins'].min()}")
print(f"Máximo: {df['monthly_watch_time_mins'].max()}")
print(f"Media: {df['monthly_watch_time_mins'].mean():.2f}")

# Aplicar filtro (0 a 5000 minutos)
df = df[(df['monthly_watch_time_mins'] >= 0) & (df['monthly_watch_time_mins'] <= 5000)]

# Verificar resultado
print("\n=== DESPUÉS DEL FILTRO ===\n")
print(f"Total: {len(df)}")
print(f"Mínimo: {df['monthly_watch_time_mins'].min()}")
print(f"Máximo: {df['monthly_watch_time_mins'].max()}")
print(f"Media: {df['monthly_watch_time_mins'].mean():.2f}")

log_data.append({ "Paso": "04", "Descripción": "filtrar los minutos de visualización al mes (0-5000 minutos)", "Filas": df.shape[0], "Nulos": int(df.isnull().sum().sum()), "Retención (%)": round((df.shape[0] / 18000) * 100, 2)
})

df.to_json('../data/processed/streaming_users_clean.json', 
           orient='records', 
           indent=2,
           date_format='iso',
           force_ascii=False)

log_df = pd.DataFrame(log_data)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

=== ESTADO ACTUAL ===

Total: 7601
Nulos: 184
Mínimo: -120.0
Máximo: 99999.0
Media: 1091.37

=== DESPUÉS DEL FILTRO ===

Total: 7342
Mínimo: 0.0
Máximo: 4193.7
Media: 796.54


Estandarizacion de la columna de generos favoritos

Este bloque de código se encarga de limpiar y unificar los valores de la columna "favorite_genre" del dataset. Primero reemplaza los valores nulos por el texto "Sin género favorito", para que no queden espacios vacíos en los datos. Después define un diccionario llamado mapeo_generos que traduce todas las variantes posibles de un mismo género (mayúsculas, minúsculas, en inglés o mal escritas, como "thriler") hacia una única forma estandarizada y bien escrita en español.

In [72]:
import pandas as pd

# Cargar dataset
df = pd.read_json('../data/processed/streaming_users_clean.json')

# PASO 1: Reemplazar nulos con "Sin género favorito"
df['favorite_genre'] = df['favorite_genre'].fillna('Sin género favorito')

# PASO 2: Limpiar espacios en blanco
df['favorite_genre'] = df['favorite_genre'].str.strip()

# PASO 3: Crear mapeo de géneros (¡ANTES de usarlo!)
mapeo_generos = {
    'drama': 'Drama', 'Drama': 'Drama', 'DRAMA': 'Drama',
    'comedia': 'Comedia', 'Comedia': 'Comedia', 'COMEDIA': 'Comedia', 'comedy': 'Comedia', 'Comedy': 'Comedia', 'COMEDY': 'Comedia',
    'thriller': 'Thriller', 'Thriller': 'Thriller', 'THRILLER': 'Thriller', 'thriler': 'Thriller',
    'acción': 'Acción', 'Acción': 'Acción', 'ACCIÓN': 'Acción', 'accion': 'Acción', 'Accion': 'Acción', 'Action': 'Acción', 'action': 'Acción',
    'romance': 'Romance', 'Romance': 'Romance', 'ROMANCE': 'Romance',
    'documental': 'Documental', 'Documental': 'Documental', 'DOCUMENTAL': 'Documental', 'Documentary': 'Documental', 'documentary': 'Documental', 'DOC': 'Documental',
    'crime': 'Crimen', 'Crime': 'Crimen', 'CRIME': 'Crimen', 'crimen': 'Crimen', 'Crimen': 'Crimen', 'CRIMEN': 'Crimen',
    'Sin género favorito': 'Sin género favorito',
}

# PASO 4: Aplicar el mapeo (convertir a minúsculas para mejor matching)
df['favorite_genre'] = df['favorite_genre'].str.lower().map(mapeo_generos).fillna(df['favorite_genre'].str.title())

# PASO 5: Verificar resultados
print("\n=== VERIFICACIÓN DESPUÉS DE LA TRANSFORMACIÓN ===\n")
nulos_restantes = df['favorite_genre'].isnull().sum()
print(f"Nulos restantes en favorite_genre: {nulos_restantes}")

# Mostrar valores únicos después de estandarizar
print("\n=== VALORES ÚNICOS EN favorite_genre (DESPUÉS) ===\n")
print(f"Total de valores únicos: {df['favorite_genre'].nunique()}")
print(f"\nValores después de estandarizar:\n{sorted(df['favorite_genre'].unique())}")

# PASO 6: Guardar
df.to_json('../data/processed/streaming_users_clean.json', 
           orient='records', 
           indent=2,
           force_ascii=False)

# PASO 7: Actualizar log
log_data.append({ 
    "Paso": "05", 
    "Descripción": "Reemplazar nulos en favorite_genre y estandarizar géneros con mapeo", 
    "Filas": df.shape[0], 
    "Nulos": int(df.isnull().sum().sum()), 
    "Retención (%)": round((df.shape[0] / 18000) * 100, 2)
})

log_df = pd.DataFrame(log_data)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

print("\n✅ Géneros estandarizados correctamente")


=== VERIFICACIÓN DESPUÉS DE LA TRANSFORMACIÓN ===

Nulos restantes en favorite_genre: 0

=== VALORES ÚNICOS EN favorite_genre (DESPUÉS) ===

Total de valores únicos: 9

Valores después de estandarizar:
['Acción', 'Comedia', 'Crimen', 'Doc', 'Documental', 'Drama', 'Romance', 'Sin Género Favorito', 'Thriller']

✅ Géneros estandarizados correctamente


Filtro por edad

Este bloque de código se encarga de limpiar la columna "age" del dataset, sacando los registros que tienen edades poco realistas. Hhace un diagnóstico rápido, mostrando la edad mínima y máxima actuales, y contando cuántos registros caen fuera del rango considerado válido (menor a 4 años o mayor a 95), además de mostrar cuáles son esos valores atípicos para que puedas revisarlos antes de filtrar. Después aplica el filtro propiamente dicho, quedándose solo con las filas donde age está entre 4 y 95

In [73]:
df = pd.read_json('../data/processed/streaming_users_clean.json')

# PASO 1: Verificar edades fuera de rango
print("=== VERIFICACIÓN DE EDADES ===\n")
print(f"Total de registros: {len(df)}")
print(f"Edad mínima: {df['age'].min()}")
print(f"Edad máxima: {df['age'].max()}")

# Identificar edades fuera de rango (menor a 4 o mayor a 95)
edades_fuera_rango = df[(df['age'] < 4) | (df['age'] > 95)]
print(f"\nRegistros con edad fuera de rango (4-95): {len(edades_fuera_rango)}")

if len(edades_fuera_rango) > 0:
    print("\nEjemplos de edades fuera de rango:")
    
    print(f"\nEdades fuera de rango encontradas: {edades_fuera_rango['age'].unique()}")

# PASO 2: Filtrar manteniendo solo edades entre 4 y 95 
df = df[(df['age'] >= 4) & (df['age'] <= 95)]

# PASO 3: Verificar después del filtro 
print("\n=== DESPUÉS DEL FILTRO ===\n")
print(f"Total de registros: {len(df)}")
print(f"Edad mínima: {df['age'].min()}")
print(f"Edad máxima: {df['age'].max()}")
print(f"Registros eliminados: {len(edades_fuera_rango)}")

df.to_json('../data/processed/streaming_users_clean.json', 
           orient='records', 
           indent=2,
           force_ascii=False)

log_data.append({ 
    "Paso": "06", "Descripción": "Filtrar edades (4-95 años)", "Filas": df.shape[0], "Nulos": int(df.isnull().sum().sum()), "Retención (%)": round((df.shape[0] / 18000) * 100, 2)
})

log_df = pd.DataFrame(log_data)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

=== VERIFICACIÓN DE EDADES ===

Total de registros: 7342
Edad mínima: -5
Edad máxima: 150

Registros con edad fuera de rango (4-95): 91

Ejemplos de edades fuera de rango:

Edades fuera de rango encontradas: [  0 130 150  -5]

=== DESPUÉS DEL FILTRO ===

Total de registros: 7251
Edad mínima: 4
Edad máxima: 80
Registros eliminados: 91


Estandarización de la columna de paises

Este bloque de código toma el dataset y estandariza la columna "country", que venía con inconsistencias típicas de carga manual: mayúsculas y minúsculas mezcladas, códigos ISO (ARG, BRA, CHL), espacios de más y nombres en inglés o español distintos para el mismo país. Para esto arma un diccionario (mapeo_paises) que traduce todas esas variantes a una única forma canónica en minúsculas.

In [74]:
df = pd.read_json('../data/processed/streaming_users_clean.json')

# PASO 1: Crear mapeo de países (COMPLETO)
mapeo_paises = {
    'argentina': 'argentina', 'Argentina': 'argentina', 'arg': 'argentina', 'ARG': 'argentina', 'argentina ': 'argentina',
    'brasil': 'brasil', 'Brasil': 'brasil', 'bra': 'brasil', 'BRA': 'brasil', 'brazil': 'brasil', 'Brazil': 'brasil',
    'chile': 'chile', 'Chile': 'chile', 'chl': 'chile', 'CHL': 'chile', 'chile ': 'chile',
    'colombia': 'colombia', 'Colombia': 'colombia', 'col': 'colombia', 'COL': 'colombia',
    'méxico': 'méxico', 'México': 'méxico', 'mexico': 'méxico', 'Mexico': 'méxico', 'mex': 'méxico', 'MEX': 'méxico',
    'perú': 'perú', 'Perú': 'perú', 'peru': 'perú', 'Peru': 'perú', 'per': 'perú', 'PER': 'perú',
    'uruguay': 'uruguay', 'Uruguay': 'uruguay', 'ury': 'uruguay', 'URY': 'uruguay'
}

# PASO 2: Limpiar espacios, convertir a minúsculas y aplicar mapeo
df['country'] = df['country'].str.strip().str.lower().map(mapeo_paises).fillna(df['country'].str.strip().str.lower())

# PASO 3: Verificar resultados
print("\n=== VALORES ÚNICOS EN country ===\n")
valores_unicos_despues = df['country'].unique()
print(f"Total de valores únicos: {len(valores_unicos_despues)}")
print(f"\nValores después de estandarizar:\n{sorted(valores_unicos_despues)}")

print("\n=== FRECUENCIA DE PAÍSES ===\n")
print(df['country'].value_counts().sort_index())

# PASO 4: Guardar
df.to_json('../data/processed/streaming_users_clean.json', 
           orient='records', 
           indent=2,
           force_ascii=False)

log_data.append({ 
    "Paso": "07", 
    "Descripción": "Estandarizar países (mapeo completo)", 
    "Filas": df.shape[0], 
    "Nulos": int(df.isnull().sum().sum()), 
    "Retención (%)": round((df.shape[0] / 18000) * 100, 2)
})

log_df = pd.DataFrame(log_data)
log_df.to_csv('../logs/pipeline_log.csv', index=False)


=== VALORES ÚNICOS EN country ===

Total de valores únicos: 7

Valores después de estandarizar:
['argentina', 'brasil', 'chile', 'colombia', 'méxico', 'perú', 'uruguay']

=== FRECUENCIA DE PAÍSES ===

country
argentina    1011
brasil       1047
chile        1059
colombia     1048
méxico       1041
perú         1023
uruguay      1022
Name: count, dtype: int64


Estandarización de la columna suscription_plan

Este bloque se encarga de limpiar y unificar los valores de la columna subscription_plan, que llegó con inconsistencias típicas de carga manual (mayúsculas y minúsculas mezcladas, espacios de más, variantes en inglés y español, y hasta errores de tipeo como "Premiun")

In [75]:
df = pd.read_json('../data/processed/streaming_users_clean.json')

# PASO 1: Verificar valores únicos actuales
print("=== VERIFICACIÓN DE subscription_plan ===\n")
print(f"Total de registros: {len(df)}")
print(f"\nValores únicos actuales ({df['subscription_plan'].nunique()}):")
print(sorted(df['subscription_plan'].unique()))

df['subscription_plan'] = df['subscription_plan'].str.strip().map(mapeo_paises).fillna(df['subscription_plan']).str.lower()

# PASO 2: Crear mapeo de planes
mapeo_planes = {
    # Básico y todas sus variantes
    'básico': 'Básico', 'Básico': 'Básico', 'BASICO': 'Básico', 'basico': 'Básico', 'Basic': 'Básico', 'basic': 'Básico', 'BASIC': 'Básico', 'básico ': 'Básico', 'Basico': 'Básico', 'BASICO': 'Básico', 'básico  ': 'Básico',
    'estándar': 'Estándar', 'Estándar': 'Estándar', 'ESTÁNDAR': 'Estándar', 'estandar': 'Estándar', 'Estandar': 'Estándar', 'Standard': 'Estándar', 'standard': 'Estándar', 'STANDARD': 'Estándar', 'Std': 'Estándar', 'std': 'Estándar', 'estándar ': 'Estándar', 'Estandar': 'Estándar', 'ESTANDAR': 'Estándar', 'estándar  ': 'Estándar',
    'premium': 'Premium', 'Premium': 'Premium', 'PREMIUM': 'Premium', 'Premiun': 'Premium', 'premiun': 'Premium', 'premium ': 'Premium', 'Premium ': 'Premium','Premium  ': 'Premium', 'premium  ': 'Premium', 'PREMIUM ': 'Premium', 'Premiun ': 'Premium',
}

# PASO 3: Aplicar el mapeo
# Primero limpiar espacios en blanco
df['subscription_plan'] = df['subscription_plan'].str.strip()

# Aplicar mapeo (convertir a minúsculas para mejor matching)
df['subscription_plan'] = df['subscription_plan'].str.lower().map(mapeo_planes).fillna(df['subscription_plan'].str.title())

# PASO 4: Verificar resultados
print("\n=== DESPUÉS DE LA ESTANDARIZACIÓN ===\n")
print(f"Valores únicos después ({df['subscription_plan'].nunique()}):")
print(sorted(df['subscription_plan'].unique()))

# PASO 5: Verificar que no queden valores sin mapear
planes_esperados = ['Básico', 'Estándar', 'Premium']
planes_encontrados = df['subscription_plan'].unique()

df.to_json('../data/processed/streaming_users_clean.json', 
           orient='records', 
           indent=2,
           force_ascii=False)

log_data.append({ 
    "Paso": "08", "Descripción": "Estandarización de subscription_plan a (Básico/Estándar/Premium)", "Filas": df.shape[0], "Nulos": int(df.isnull().sum().sum()), "Retención (%)": round((df.shape[0] / 18000) * 100, 2)
})

log_df = pd.DataFrame(log_data)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

=== VERIFICACIÓN DE subscription_plan ===

Total de registros: 7251

Valores únicos actuales (15):
['BASICO', 'Basic', 'Básico', 'Estándar', 'Estándar ', 'PREMIUM', 'Premium', 'Premium ', 'Premiun', 'STANDARD', 'Std', 'basico', 'básico', 'estandar', 'premium']

=== DESPUÉS DE LA ESTANDARIZACIÓN ===

Valores únicos después (3):
['Básico', 'Estándar', 'Premium']
